In [1]:
import slangpy as spy
import pathlib
import numpy as np

In [2]:
np.random.seed(348)

In [3]:
device = spy.create_device(
    enable_print=True,
    include_paths=[
        pathlib.Path(".").absolute()
    ]
)
module = spy.Module.load_from_file(device, "slang/particle.slang")
module

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


slangpy.Module("slang/particle.slang", path="/home/fangjun/stanford/gslang/notebooks/playgrounds/slang/particle.slang")

**NOTE**: This notebook doesn't support Metal at this time due to https://github.com/shader-slang/slangpy/issues/180#issuecomment-2864789137

In [4]:
assert device.info.type != spy.DeviceType.metal

In [5]:
# Init particle buffer.
particle_buf = spy.InstanceList(
    struct=module.Particle.as_struct(),
    data={
        "position": spy.NDBuffer(device, dtype=spy.float3, shape=(16,)),
        "velocity": spy.NDBuffer(device, dtype=spy.float3, shape=(16,))
    }
)
particle_buf

In [6]:
particle_buf.info(spy.call_id())
print(device.flush_print_to_string())

Particle 0: position={0, 0, 0}, velocity={0, 0, 0}
Particle 1: position={0, 0, 0}, velocity={0, 0, 0}
Particle 2: position={0, 0, 0}, velocity={0, 0, 0}
Particle 3: position={0, 0, 0}, velocity={0, 0, 0}
Particle 4: position={0, 0, 0}, velocity={0, 0, 0}
Particle 5: position={0, 0, 0}, velocity={0, 0, 0}
Particle 6: position={0, 0, 0}, velocity={0, 0, 0}
Particle 7: position={0, 0, 0}, velocity={0, 0, 0}
Particle 8: position={0, 0, 0}, velocity={0, 0, 0}
Particle 9: position={0, 0, 0}, velocity={0, 0, 0}
Particle 10: position={0, 0, 0}, velocity={0, 0, 0}
Particle 11: position={0, 0, 0}, velocity={0, 0, 0}
Particle 12: position={0, 0, 0}, velocity={0, 0, 0}
Particle 13: position={0, 0, 0}, velocity={0, 0, 0}
Particle 14: position={0, 0, 0}, velocity={0, 0, 0}
Particle 15: position={0, 0, 0}, velocity={0, 0, 0}



In [7]:
# Load from numpy.
position = np.random.randn(16, 3).astype(np.float32)
velocity = np.random.randn(16, 3).astype(np.float32)
particle_buf.get_this()["position"].copy_from_numpy(position)
particle_buf.get_this()["velocity"].copy_from_numpy(velocity)

In [8]:
particle_buf.info(spy.call_id())
print(device.flush_print_to_string())

Particle 0: position={-0.74514765, -1.4626689, 1.5440476}, velocity={-1.0369624, -0.09391204, 2.5216737}
Particle 1: position={-0.3148266, 1.3637321, 0.32085106}, velocity={1.6197588, -0.8202258, -0.48772755}
Particle 2: position={1.29413, 0.014315604, 0.14287208}, velocity={-0.09869889, -1.8365425, 0.5553512}
Particle 3: position={-0.41008282, -1.1208322, -0.55690503}, velocity={-0.8800851, 0.7770424, 1.7593306}
Particle 4: position={1.6357784, -1.0389293, -0.29886138}, velocity={-0.7572701, 0.25917792, 0.50515294}
Particle 5: position={-0.39862242, -0.33556923, -1.24802}, velocity={-0.42478323, 0.15291093, 0.4785611}
Particle 6: position={-0.6731114, 0.607563, 0.8623601}, velocity={0.71549106, 0.83473426, -0.48329803}
Particle 7: position={-1.7406042, 0.29265207, -0.66594994}, velocity={-0.31592175, -0.94085723, 0.043877386}
Particle 8: position={0.6454373, -1.2618792, 0.7806917}, velocity={0.9080636, 0.81681174, -1.606322}
Particle 9: position={0.86589104, -0.7342854, -0.9310714}, v

In [9]:
# Test mutating method.
particle_buf.update(4.2)

In [10]:
res = particle_buf.get_this()["position"].to_numpy()
ref = position + velocity * 4.2
np.linalg.norm(res - ref)

np.float32(6.6442146e-07)